<a href="https://colab.research.google.com/github/stevenolanecon/7002LBSAI/blob/main/notebooks/week5_data_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Before you start:**

- Go to **File → Save a copy in Drive**. Until you do this, your work only exists in this browser tab – you can't save changes back to GitHub, and closing the tab or losing the session will lose your work.
- Turn off Colab's autocomplete: **Tools → Settings → Editor → uncheck "Show context-powered code completions."** These exercises are meant to be worked through yourself – treat this the same as switching off a calculator's solver mode in an exam. It's a setting on your own Google account, not something built into this notebook, so you'll need to do it once per account.

# Week 5 – Data Exercises: Generalising a Regression

These exercises are adapted from Bekes & Kezdi, *Data Analysis for Business, Economics, and Policy* – the textbook this module follows – and focus specifically on this week's inference content: standard errors and confidence intervals for a slope, robust standard errors, testing whether a slope is really different from zero, and the difference between a confidence interval and a prediction interval. There's no output shown here to check yourself against: the point is to practise these skills on real data you haven't seen the answer for.

Easier and/or shorter exercises are marked **[\*]**; harder and/or longer exercises are marked **[\*\*]**.

Two of these four are adapted directly from the textbook's own chapter exercises (Q1 and Q2 below); the other two are original, built to cover two things this week's workshop teaches that the textbook's own exercises for this chapter don't actually test (robust standard errors, and the CI-vs-prediction-interval distinction).

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

hotels_vienna_path = "https://raw.githubusercontent.com/stevenolanecon/7002LBSAI/main/data/hotels_vienna.csv"
hotels_europe_path = "https://raw.githubusercontent.com/stevenolanecon/7002LBSAI/main/data/hotels_europe_nov2017_weekday.csv"
used_cars_path = "https://raw.githubusercontent.com/stevenolanecon/7002LBSAI/main/data/used_cars_chicago.csv"
wms_path = "https://raw.githubusercontent.com/stevenolanecon/7002LBSAI/main/data/wms_management_2006.csv"

vienna = pd.read_csv(hotels_vienna_path).dropna(subset=['rating'])
europe = pd.read_csv(hotels_europe_path)
cars = pd.read_csv(used_cars_path)
wms = pd.read_csv(wms_path)

## Question 1 [\*]

Use the `europe` dataset and pick a European city (not Vienna – you've already looked at that one). Select a sample appropriate for a couple searching for a good deal: 3-to-4-star hotels, not very far from the centre (you decide what counts as "not very far" and say why).

1. Show a LOWESS regression for the pattern between **log hotel price** and **rating**.
2. Estimate a simple linear regression and at least one alternative (e.g. a quadratic) that might fit the pattern better.
3. Visualise the confidence interval around each regression line, together with the LOWESS curve, and decide which specification is the best approximation.

> **Syntax hint – confidence interval around a regression line.** `seaborn`'s `regplot` draws the fitted line together with its 95% confidence band automatically – `order=1` fits a straight line, `order=2` fits a quadratic:
>
> ```python
> sns.regplot(x='x', y='y', data=df, order=1, ci=95)   # linear, with CI band
> sns.regplot(x='x', y='y', data=df, order=2, ci=95)   # quadratic, with CI band
> ```
>
> This is a different thing from the LOWESS hint you used in Week 4 (`sns.regplot(..., lowess=True)`) – LOWESS doesn't impose a functional form, so `seaborn` doesn't draw a CI band around it. Plot the LOWESS curve and the parametric fit(s) on the same axes to compare them directly.

*Which specification do you think is the best approximation, and why?*

## Question 2 [\*\*]

Use the `wms` dataset – survey-based management-quality scores for firms across 13 countries in 2006 (`management`, scored 1–5) – and pick a country from the `country` column (check the sample size for your chosen country first; some are much smaller than others).

1. Estimate a linear regression of `management` on log-employment (`emp_firm`, logged). Interpret the slope coefficient and its 95% confidence interval.
2. Explore whether the pattern of association is actually linear, using a LOWESS regression.
3. Estimate a regression that can capture any nonlinearity you find (e.g. add a squared term).
4. Carry out a test to see whether you can reject that the linear approximation was good enough for this country's data.

> **Syntax hint – testing a single coefficient.** You've already seen the 95% CI columns in `.summary()` output – the same table also gives you everything you need for part 4: a coefficient's own `P>|t|` column *is* the hypothesis test that it's equal to zero. For the squared term specifically, a p-value below 0.05 is evidence against "the linear model was good enough" – the curvature is real, not just noise. Comparing AIC between the linear and quadratic specifications (as in Week 4) is a reasonable second way to answer the same question.

*Your interpretation of the slope and its CI, and your conclusion about whether linear was good enough:*

## Question 3 [\*]

Use the `cars` dataset (the same Toyota Camry data from Week 4). Estimate a simple linear regression of `price` on `age`.

1. Report the classical (non-robust) standard error for the slope on `age`.
2. Now refit the same regression with heteroskedasticity-robust (HC1) standard errors, and report the slope's standard error again.
3. Are the two standard errors meaningfully different here? What would you conclude about whether heteroskedasticity is a real concern for this regression – and which standard error would you report in a write-up?

> **Syntax hint – robust standard errors.** Add a `cov_type` argument to `.fit()`:
>
> ```python
> model_classical = smf.ols('y ~ x', data=df).fit()
> model_robust = smf.ols('y ~ x', data=df).fit(cov_type='HC1')
> ```
>
> Both give you the same coefficients – robust SEs don't change *what* you estimated, only how much you trust the precision of the estimate. Compare `model_classical.bse` and `model_robust.bse`.

*Your comparison of the two standard errors, and which you'd report:*

## Question 4 [\*]

Use the `vienna` dataset. Estimate a simple linear regression of `price` on `rating`.

1. For a hotel with `rating` = 4.0, compute **both** the 95% confidence interval for the *average* price of hotels at that rating, **and** the 95% prediction interval for the price of *one specific* hotel with that rating.
2. Which interval is wider, and by roughly how much?
3. In your own words: what's the difference between the two questions each interval is answering? Why does that difference explain the gap in width?

> **Syntax hint – confidence interval vs. prediction interval.** Once you've fit a model, `get_prediction()` on new data returns both intervals in one table:
>
> ```python
> model = smf.ols('y ~ x', data=df).fit()
> new_x = pd.DataFrame({'x': [SOME_VALUE]})
> pred = model.get_prediction(new_x).summary_frame(alpha=0.05)
> print(pred)
> ```
>
> `mean_ci_lower` / `mean_ci_upper` are the confidence interval (for the *average* y at that x); `obs_ci_lower` / `obs_ci_upper` are the prediction interval (for *one new* observation at that x).

*Your comparison of the two intervals, and your explanation of why they differ:*